# 2.3 一條長卡怎樣混成新的特徵？


拼接保留了各字特徵，但要猜下一字，還需要讓不同格共同影響結果。做法像配方：各輸入乘自己的權重，再加總；每套配方產生一個新數字。

先把長卡縮成三格 `[1,2,4]`，手算兩套：

| 配方 | 三個乘數 | 最後再加的偏移 | 輸出 |
| --- | --- | ---: | --- |
| 第一套 | `[1,0,2]` | 1 | `1×1+2×0+4×2+1=10` |
| 第二套 | `[0,-1,1]` | 0 | `1×0+2×(-1)+4×1=2` |

乘數叫**權重**，加總後再加的數叫**偏移**。它們都可調；本節先指定，看看程式是否得到 `[10,2]`。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from torch import nn

layer = nn.Linear(3, 2)
with torch.no_grad():
    layer.weight.copy_(torch.tensor([[1.0, 0.0, 2.0], [0.0, -1.0, 1.0]]))
    layer.bias.copy_(torch.tensor([1.0, 0.0]))
x = torch.tensor([[1.0, 2.0, 4.0]])
y = layer(x)
manual = x @ layer.weight.T + layer.bias
print(layer.weight.shape, y)
assert torch.allclose(y, manual)

torch.Size([2, 3]) tensor([[10.,  2.]], grad_fn=<AddmmBackward0>)


`nn.Linear(3,2)` 建立三格輸入、兩格輸出的線性層。`layer.weight` 是 `[2,3]`：每列一套配方，每套三個乘數；`bias` 則儲存兩個偏移。手動填入後，`layer(x)` 得到 `[[10,2]]`。

`manual=x @ layer.weight.T+layer.bias` 是同一運算的展開。`.T` 交換權重的列欄，使 `[1,3] @ [3,2]` 能把三格相乘加總，留下兩個輸出。若忘記轉置，這個非方形例子會直接顯示尺寸不符；方形表即使寫錯，也可能不報錯而算出另一套配方。

Linear 只沿最後一個特徵軸計算。輸入若是 `[2,4,3]`，它會讓每筆、每位置共用這兩套配方，輸出 `[2,4,2]`。若要混合整段前文，就先像上一節拼成長卡，再讓輸入格數等於 C×D；不會把不同筆材料混成一筆。

練習把第一套第三個乘數由 2 改成 3，第一輸出增加 4，第二不變，應得 `[[14,2]]`。這個具體差額比只看形狀，更能指出哪個參數影響哪個結果。

<details>
<summary>補充與重做</summary>

加權和與矩陣見 [W.5](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)。訓練會依答案代價調整 `weight` 和 `bias`，見 [1.11](https://birdhackor.github.io/tiny-perceptron-vlm/1.11.html)、[1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)；本例數字由人指定，沒有學習或能力評測。

</details>
